# Sidebar Links Checker

This notebook ensures that all sidebar links are properly linked to a page in the `/templates` directory, creating any missing pages as needed.

## Import Required Libraries

Import necessary libraries such as os and json for file and directory operations.

In [ ]:
import os
import json
from pathlib import Path
import datetime

## Load Sidebar Links

Load the sidebar configuration file (e.g., sidebar.json) and extract all links.

In [ ]:
# Define paths
project_root = Path('d:/Projects/impressioncore')
sidebar_path = project_root / 'config' / 'sidebar.json'
templates_dir = project_root / 'templates'

# Ensure templates directory exists
if not templates_dir.exists():
    templates_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created templates directory at {templates_dir}")

# Load the sidebar configuration
try:
    with open(sidebar_path, 'r') as f:
        sidebar_config = json.load(f)
    print(f"Successfully loaded sidebar configuration from {sidebar_path}")
except FileNotFoundError:
    print(f"Sidebar configuration not found at {sidebar_path}. Creating a sample one.")
    # Create a sample sidebar config for demonstration
    sidebar_config = {
        "links": [
            {"title": "Home", "url": "/home"},
            {"title": "About", "url": "/about"},
            {"title": "Services", "url": "/services"},
            {"title": "Contact", "url": "/contact"}
        ]
    }
    
    # Create the config directory if it doesn't exist
    config_dir = project_root / 'config'
    if not config_dir.exists():
        config_dir.mkdir(parents=True, exist_ok=True)
    
    # Save the sample sidebar config
    with open(sidebar_path, 'w') as f:
        json.dump(sidebar_config, f, indent=2)
    print(f"Created sample sidebar configuration at {sidebar_path}")

# Extract all links from the sidebar configuration
sidebar_links = []

def extract_links(config_item):
    if isinstance(config_item, dict):
        if 'url' in config_item and config_item['url']:
            sidebar_links.append(config_item['url'])
        if 'children' in config_item and isinstance(config_item['children'], list):
            for child in config_item['children']:
                extract_links(child)
    elif isinstance(config_item, list):
        for item in config_item:
            extract_links(item)

# Extract links depending on the structure of the sidebar config
if 'links' in sidebar_config and isinstance(sidebar_config['links'], list):
    extract_links(sidebar_config['links'])
else:
    extract_links(sidebar_config)

# Clean up the links (remove leading slash, add .html extension if missing)
cleaned_links = []
for link in sidebar_links:
    # Remove leading slash
    link = link.lstrip('/')
    # Add .html extension if missing and not empty
    if link and not link.endswith('.html'):
        link = f"{link}.html"
    if link:  # Only add non-empty links
        cleaned_links.append(link)

print(f"Found {len(cleaned_links)} links in sidebar configuration:")
for link in cleaned_links:
    print(f"  - {link}")

## Check for Missing Pages

Iterate through the links and check if the corresponding files exist in the `/templates` directory.

In [ ]:
# Check if each link has a corresponding file in the templates directory
existing_pages = []
missing_pages = []

for link in cleaned_links:
    # Build the expected file path
    expected_file = templates_dir / link
    
    # Check if the file exists
    if expected_file.exists():
        existing_pages.append(link)
    else:
        missing_pages.append(link)

print(f"Found {len(existing_pages)} existing pages:")
for page in existing_pages:
    print(f"  ✅ {page}")

print(f"\nIdentified {len(missing_pages)} missing pages:")
for page in missing_pages:
    print(f"  ❌ {page}")

## Create Missing Pages

For each missing page, create a new file in the `/templates` directory with a placeholder template.

In [ ]:
# Define a template for missing pages
def create_template(title):
    return f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
    <link rel="stylesheet" href="/static/css/main.css">
</head>
<body>
    <div class="container">
        <h1>{title}</h1>
        <p>This is a placeholder page for {title}. Content to be added.</p>
        <p>Created on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    </div>
    <script src="/static/js/main.js"></script>
</body>
</html>
"""

# Create missing pages
created_pages = []
failed_pages = []

for page in missing_pages:
    try:
        # Get the title from the page path
        title = page.split('/')[-1].replace('.html', '').replace('_', ' ').title()
        
        # Create the directory if it doesn't exist
        page_path = templates_dir / page
        page_dir = page_path.parent
        if not page_dir.exists():
            page_dir.mkdir(parents=True, exist_ok=True)
        
        # Create the page
        with open(page_path, 'w') as f:
            f.write(create_template(title))
        
        created_pages.append(page)
        print(f"Created page: {page}")
    except Exception as e:
        failed_pages.append((page, str(e)))
        print(f"Failed to create page {page}: {e}")

print(f"\nCreated {len(created_pages)} missing pages")
if failed_pages:
    print(f"Failed to create {len(failed_pages)} pages:")
    for page, error in failed_pages:
        print(f"  - {page}: {error}")

## Update Sidebar Links

Ensure all sidebar links are updated to point to the correct files in the `/templates` directory.

In [ ]:
# This function updates the sidebar config if needed
def update_sidebar_links(config_item):
    if isinstance(config_item, dict):
        if 'url' in config_item and config_item['url']:
            link = config_item['url'].lstrip('/')
            if link and not link.endswith('.html'):
                config_item['url'] = f"/{link}.html"
        if 'children' in config_item and isinstance(config_item['children'], list):
            for child in config_item['children']:
                update_sidebar_links(child)
    elif isinstance(config_item, list):
        for item in config_item:
            update_sidebar_links(item)

# Make a copy of the original config for comparison
original_config = json.dumps(sidebar_config, indent=2)

# Update the links in the config
if 'links' in sidebar_config and isinstance(sidebar_config['links'], list):
    update_sidebar_links(sidebar_config['links'])
else:
    update_sidebar_links(sidebar_config)

# Check if the config was updated
updated_config = json.dumps(sidebar_config, indent=2)
if original_config != updated_config:
    print("Sidebar configuration was updated. Saving changes...")
    with open(sidebar_path, 'w') as f:
        f.write(updated_config)
    print(f"Saved updated sidebar configuration to {sidebar_path}")
else:
    print("No updates were needed for the sidebar configuration.")

## Generate Report of Link Status

Generate a report listing which links are complete and working, and which links were missing and created.

In [ ]:
# Generate a detailed report
report = {
    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_links": len(cleaned_links),
    "existing_pages": existing_pages,
    "created_pages": created_pages,
    "failed_pages": [p[0] for p in failed_pages]
}

# Save the report to a file
report_path = project_root / 'reports' / 'sidebar_links_report.json'

# Create reports directory if it doesn't exist
reports_dir = report_path.parent
if not reports_dir.exists():
    reports_dir.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"Report saved to {report_path}")

# Display a summary of the report
print("\n===== SIDEBAR LINKS REPORT =====")
print(f"Report generated on: {report['timestamp']}")
print(f"Total sidebar links: {report['total_links']}")
print(f"Existing pages: {len(report['existing_pages'])}")
print(f"Pages created: {len(report['created_pages'])}")
print(f"Failed to create: {len(report['failed_pages'])}")
print("===============================")

# Verify all pages now exist
all_pages_exist = True
for link in cleaned_links:
    if not (templates_dir / link).exists():
        all_pages_exist = False
        print(f"WARNING: Page still missing: {link}")

if all_pages_exist:
    print("SUCCESS: All sidebar links now have corresponding pages in the templates directory.")
else:
    print("WARNING: Some sidebar links still don't have corresponding pages.")

# Sidebar Links Checker

This notebook checks if all sidebar links have corresponding pages in the `/templates` directory and creates any missing pages as needed.

In [ ]:
# Import Required Libraries
import os
import json
import datetime

# For generating basic content for missing pages
from jinja2 import Template

## Load Sidebar Links

We'll load the sidebar configuration file and extract all links. This assumes the sidebar configuration is stored in a JSON file.

In [ ]:
# Define paths
TEMPLATES_DIR = '../templates'
SIDEBAR_CONFIG_PATH = '../config/sidebar.json'

# Function to load sidebar configuration
def load_sidebar_config(config_path):
    try:
        with open(config_path, 'r') as f:
            config = json.load(f)
        print(f"Sidebar configuration loaded from {config_path}")
        return config
    except FileNotFoundError:
        print(f"Sidebar configuration file not found at {config_path}")
        return None
    except json.JSONDecodeError:
        print(f"Invalid JSON in sidebar configuration file at {config_path}")
        return None

# Load the sidebar configuration
sidebar_config = load_sidebar_config(SIDEBAR_CONFIG_PATH)

# Extract all links from the sidebar configuration
def extract_links(config, links=None):
    if links is None:
        links = []
    
    if isinstance(config, dict):
        if 'href' in config and config['href'] is not None:
            links.append(config['href'])
        for key, value in config.items():
            extract_links(value, links)
    elif isinstance(config, list):
        for item in config:
            extract_links(item, links)
    
    return links

# Get all links from the sidebar
if sidebar_config:
    sidebar_links = extract_links(sidebar_config)
    print(f"Found {len(sidebar_links)} links in the sidebar configuration")
    print(sidebar_links[:5])  # Show the first few links as a preview
else:
    sidebar_links = []
    print("No sidebar links found")

## Check for Missing Pages

Now we'll check which links don't have corresponding files in the templates directory.

In [ ]:
# Function to normalize links to file paths
def normalize_link_to_file_path(link):
    # Remove leading slash if present
    if link.startswith('/'):
        link = link[1:]
    
    # Handle index pages
    if link == '' or link == '/':
        return os.path.join(TEMPLATES_DIR, 'index.html')
    
    # Handle links without extensions
    if not link.endswith('.html'):
        link = f"{link}.html"
    
    # Construct the full path
    return os.path.join(TEMPLATES_DIR, link)

# Check if each link has a corresponding file
missing_pages = []
existing_pages = []

for link in sidebar_links:
    file_path = normalize_link_to_file_path(link)
    if not os.path.exists(file_path):
        missing_pages.append((link, file_path))
    else:
        existing_pages.append((link, file_path))

print(f"Found {len(missing_pages)} missing pages")
print(f"Found {len(existing_pages)} existing pages")

# Display missing pages
if missing_pages:
    print("\nMissing pages:")
    for link, file_path in missing_pages[:10]:  # Show first 10 to avoid clutter
        print(f"  - {link} -> {file_path}")
    
    if len(missing_pages) > 10:
        print(f"  ... and {len(missing_pages) - 10} more")

## Create Missing Pages

For each missing page, we'll create a new file in the templates directory with basic content.

In [ ]:
# Template for new pages
page_template = Template("""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{{ title }}</title>
    <!-- Generated by sidebar_links_checker.ipynb on {{ date }} -->
</head>
<body>
    <h1>{{ title }}</h1>
    <p>This is a placeholder page for {{ link }}. Please update with your content.</p>
</body>
</html>
""")

# Function to create directories if they don't exist
def ensure_directory_exists(file_path):
    directory = os.path.dirname(file_path)
    if directory and not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Created directory: {directory}")

# Function to create a missing page
def create_page(link, file_path):
    # Ensure the directory exists
    ensure_directory_exists(file_path)
    
    # Generate a title from the link
    title = link.split('/')[-1].replace('-', ' ').replace('_', ' ').title()
    if not title:
        title = "Home"
    
    # Render the template
    content = page_template.render(
        title=title,
        link=link,
        date=datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    )
    
    # Write the file
    with open(file_path, 'w') as f:
        f.write(content)
    
    print(f"Created page: {file_path}")

# Create pages for all missing links
created_pages = []

# Ask user if they want to create the missing pages
create_missing = input(f"Do you want to create {len(missing_pages)} missing pages? (y/n): ")

if create_missing.lower() == 'y':
    for link, file_path in missing_pages:
        try:
            create_page(link, file_path)
            created_pages.append((link, file_path))
        except Exception as e:
            print(f"Error creating page for {link}: {e}")
    
    print(f"\nCreated {len(created_pages)} missing pages")
else:
    print("No pages were created")

## Update Sidebar Links

Now we'll check if all sidebar links point to the correct file paths.

In [ ]:
# Function to check if a link is valid
def is_valid_link(link):
    file_path = normalize_link_to_file_path(link)
    return os.path.exists(file_path)

# Function to suggest corrections for invalid links
def suggest_correction(link):
    # Check if similar files exist
    potential_matches = []
    
    # Remove extension if present
    base_link = link
    if '.' in os.path.basename(link):
        base_link = os.path.splitext(link)[0]
    
    # Check for files with different extensions
    for ext in ['.html', '.htm']:
        potential_path = normalize_link_to_file_path(f"{base_link}{ext}")
        if os.path.exists(potential_path):
            potential_matches.append(potential_path)
    
    return potential_matches

# Check all links and suggest corrections
invalid_links = []
correction_suggestions = {}

for link in sidebar_links:
    if not is_valid_link(link):
        invalid_links.append(link)
        suggestions = suggest_correction(link)
        if suggestions:
            correction_suggestions[link] = suggestions

print(f"Found {len(invalid_links)} invalid links in the sidebar configuration")

if correction_suggestions:
    print("\nCorrection suggestions:")
    for link, suggestions in correction_suggestions.items():
        print(f"  - {link} -> {', '.join(suggestions)}")

## Generate Report of Link Status

Finally, we'll generate a report of the link status, including which links are working and which were created.

In [ ]:
# Generate a report
report = {
    'timestamp': datetime.datetime.now().isoformat(),
    'total_links': len(sidebar_links),
    'existing_pages': len(existing_pages),
    'missing_pages': len(missing_pages),
    'created_pages': len(created_pages),
    'invalid_links': invalid_links,
    'correction_suggestions': correction_suggestions
}

# Save the report to a file
report_file = 'sidebar_links_report.json'
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nReport saved to {report_file}")

# Display a summary table
print("\nSummary:")
print(f"{'Total links:':<25} {len(sidebar_links)}")
print(f"{'Existing pages:':<25} {len(existing_pages)}")
print(f"{'Missing pages:':<25} {len(missing_pages)}")
print(f"{'Created pages:':<25} {len(created_pages)}")
print(f"{'Invalid links:':<25} {len(invalid_links)}")

# Generate a more detailed HTML report for viewing
html_report = f"""
<html>
<head>
    <title>Sidebar Links Report - {datetime.datetime.now().strftime('%Y-%m-%d')}</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        table {{ border-collapse: collapse; width: 100%; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        th {{ background-color: #f2f2f2; }}
        .success {{ color: green; }}
        .warning {{ color: orange; }}
        .error {{ color: red; }}
    </style>
</head>
<body>
    <h1>Sidebar Links Report</h1>
    <p>Generated on {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    
    <h2>Summary</h2>
    <table>
        <tr><th>Metric</th><th>Count</th></tr>
        <tr><td>Total links</td><td>{len(sidebar_links)}</td></tr>
        <tr><td>Existing pages</td><td class="success">{len(existing_pages)}</td></tr>
        <tr><td>Missing pages</td><td class="warning">{len(missing_pages)}</td></tr>
        <tr><td>Created pages</td><td class="success">{len(created_pages)}</td></tr>
        <tr><td>Invalid links</td><td class="error">{len(invalid_links)}</td></tr>
    </table>
    
    <h2>Existing Pages</h2>
    <table>
        <tr><th>Link</th><th>File Path</th></tr>
        {"".join(f"<tr><td>{link}</td><td>{path}</td></tr>" for link, path in existing_pages)}
    </table>
    
    <h2>Created Pages</h2>
    <table>
        <tr><th>Link</th><th>File Path</th></tr>
        {"".join(f"<tr><td>{link}</td><td>{path}</td></tr>" for link, path in created_pages)}
    </table>
    
    <h2>Missing Pages</h2>
    <table>
        <tr><th>Link</th><th>Expected File Path</th></tr>
        {"".join(f"<tr><td>{link}</td><td>{path}</td></tr>" for link, path in missing_pages)}
    </table>
</body>
</html>
"""

# Save the HTML report
html_report_file = 'sidebar_links_report.html'
with open(html_report_file, 'w') as f:
    f.write(html_report)

print(f"HTML report saved to {html_report_file}")

## Conclusion

This notebook provides a comprehensive solution for ensuring that all sidebar links have corresponding pages in the templates directory. It:

1. Loads the sidebar configuration and extracts all links
2. Checks for missing pages in the templates directory
3. Creates missing pages with basic templates
4. Validates sidebar links and suggests corrections
5. Generates a detailed report of the link status

You can run this notebook periodically to ensure that your sidebar links always have corresponding pages.